# Квантование, прунинг и плотные бейзлайны

Сводка сохранённых измерений. Запуск из корня репозитория или из `results/` требует Python, pandas и matplotlib. Обучение, оценка моделей и загрузка датасетов для сводки не нужны.

[Описание сравнений](comparison_notes.md) · [Бейзлайны](../baseline/README.md) · [Исходный CV-прогон](cv.ipynb) · [Исходный LLM-прогон](transformers.ipynb)

**Правило сравнения.** Внутри каждого общего прогона фиксируем исходный чекпойнт и тест. Сравниваем качество при близком размере и явно указываем число дополнительных эпох. Для LeNet размер — байты файла `state_dict`, для SmolLM — расчётный объём полезной нагрузки. Абсолютные размеры и метрики между задачами имеют разные основания.
Числа общих прогонов читаются из [CV JSON](cv_results.json) и [LLM JSON](transformers_results.json).
Плотные бейзлайны сверяются с [baseline/results.json](../baseline/results.json).
Для дополнительного эксперимента ResNet используются исходные CSV, если они доступны.


In [1]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "baseline/results.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Откройте ноутбук из корня репозитория или results/.")

baseline = json.loads((ROOT / "baseline/results.json").read_text(encoding="utf-8"))
cv_run = json.loads((ROOT / "results/cv_results.json").read_text(encoding="utf-8"))
llm_run = json.loads((ROOT / "results/transformers_results.json").read_text(encoding="utf-8"))
cv = pd.DataFrame(cv_run["rows"])
llm = pd.DataFrame(llm_run["rows"])
cv_base = cv.loc[cv["method"].eq("FP32")].iloc[0]
llm_base = llm.loc[llm["method"].eq("FP")].iloc[0]

assert cv_run["checkpoint_sha256"] == baseline["cv"]["checkpoint_sha256"]
assert cv_run["n_test"] == baseline["cv"]["examples"]
assert cv_base["accuracy_pct"] == baseline["cv"]["accuracy_pct"]
assert cv_base["state_dict_bytes"] == baseline["cv"]["state_dict_bytes"]
assert (100 * cv["correct"] / cv_run["n_test"] - cv["accuracy_pct"]).abs().max() < 1e-8

ref = baseline["transformers"]
assert (llm_run["model"], llm_run["revision"]) == (ref["model"], ref["revision"])
assert (llm_run["context"], llm_run["stride"], llm_run["scored_token_count"]) == (
    ref["window_tokens"], ref["stride_tokens"], ref["scored_tokens"])
assert math.isclose(llm_base["ppl"], ref["perplexity"], rel_tol=1e-6)
assert llm_base["payload_estimate_bytes"] == ref["parameter_payload_bytes"]
assert cv[["method", "ratio"]].duplicated().sum() == 0
assert llm[["method", "ratio"]].duplicated().sum() == 0

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
print("Бейзлайны и протоколы сохранённых общих прогонов сверены.")
print(f"Прочитано конфигураций: LeNet — {len(cv)}, SmolLM — {len(llm)}.")

Бейзлайны и протоколы сохранённых общих прогонов сверены.
Прочитано конфигураций: LeNet — 11, SmolLM — 7.


## 1. LeNet + BatchNorm · MNIST test

Один FP32-чекпойнт, 10 000 тестовых изображений. PTQ калибруется на 512 изображениях train и конвертируется в oneDNN INT8. LAMP защищает выходной слой. Для задержки используем только CPU-замеры из общей сессии, один поток и фиксированные размеры батча.

В строках с дополнительным обучением считаем разницу и с исходным FP32, и с плотным контролем на десяти эпохах. При одинаковом расписании LAMP дообучение может изменить последующее ранжирование и конкретный набор удалённых каналов.

In [2]:
cv["size_kib"] = cv["state_dict_bytes"] / 1024
cv["saved_pct"] = 100 * (1 - cv["state_dict_bytes"] / cv_base["state_dict_bytes"])
cv["delta_fp_pp"] = cv["accuracy_pct"] - cv_base["accuracy_pct"]
dense_ft = cv.loc[cv["method"].eq("Dense + FT")].iloc[0]
cv["delta_dense_ft_pp"] = (cv["accuracy_pct"] - dense_ft["accuracy_pct"]).where(
    cv["ft_epochs"].eq(dense_ft["ft_epochs"]))
for batch in (1, 256):
    cv[f"b{batch}_ms"] = cv[f"cpu_ms_batch_{batch}"].map(lambda value: value["median"])
    cv[f"speedup_b{batch}"] = cv.loc[cv["method"].eq("FP32"), f"b{batch}_ms"].iat[0] / cv[f"b{batch}_ms"]

cv_columns = ["method", "ratio", "ft_epochs", "accuracy_pct", "delta_fp_pp",
              "delta_dense_ft_pp", "size_kib", "saved_pct", "b1_ms", "b256_ms",
              "speedup_b1", "speedup_b256", "compression_stage_s"]
print(cv[cv_columns].round(3).to_string(index=False))

ptq = cv.loc[cv["method"].eq("PTQ INT8 per-channel MinMax")].iloc[0]
oneshot = cv.loc[cv["method"].eq("LAMP one-shot") & cv["ratio"].eq(.60)].iloc[0]
iterative = cv.loc[cv["method"].eq("LAMP iterative")].iloc[0]
recovered = cv.loc[cv["method"].eq("LAMP iterative + FT")].iloc[0]
assert ptq["ft_epochs"] == oneshot["ft_epochs"] == iterative["ft_epochs"] == 0
assert recovered["ft_epochs"] == dense_ft["ft_epochs"] == 10
assert oneshot["ratio"] == iterative["ratio"] == recovered["ratio"]
assert recovered["state_dict_bytes"] == iterative["state_dict_bytes"]

                        method  ratio  ft_epochs  accuracy_pct  delta_fp_pp  delta_dense_ft_pp  size_kib  saved_pct  b1_ms  b256_ms  speedup_b1  speedup_b256  compression_stage_s
                          FP32   0.00          0         98.55         0.00                NaN   179.229      0.000  0.284    8.640       1.000         1.000                0.000
    PTQ INT8 per-tensor MinMax   0.00          0         98.49        -0.06                NaN    50.781     71.667  0.597    4.119       0.476         2.097                0.122
   PTQ INT8 per-channel MinMax   0.00          0         98.52        -0.03                NaN    56.420     68.521  0.585    4.211       0.485         2.051                0.064
PTQ INT8 per-channel Histogram   0.00          0         98.52        -0.03                NaN    56.420     68.521  0.596    4.105       0.477         2.104                0.163
                 LAMP one-shot   0.50          0         87.44       -11.11                NaN    70.979 

## 2. SmolLM-135M · WikiText-2 raw test

Один чекпойнт FP16 и токенизатор, окно 512, шаг 256, 304 828 оценённых токенов. Все общие варианты без дообучения. Основная пара — RTN W8 и пятишаговый Magnitude, оба в MLP. W4 по всем Linear внутри блоков дополнительно меняет охват. 

Размер RTN включает коды, масштабы, zero points и неизменённые тензоры. PPL RTN измерена после восстановления плотных весов. Время вычисления PPL не используется как задержка низкобитного инференса.

In [3]:
llm["size_mib_est"] = llm["payload_estimate_bytes"] / (1024 ** 2)
llm["saved_pct_est"] = 100 * (1 - llm["payload_estimate_bytes"] / llm_base["payload_estimate_bytes"])
llm["removed_params_pct"] = 100 * (1 - llm["params"] / llm_base["params"])
llm["ppl_over_fp"] = llm["ppl"] / llm_base["ppl"]
llm["delta_nll"] = llm["ppl_over_fp"].map(math.log)
llm["ft_epochs"] = 0
print(llm[["method", "ratio", "bits", "ft_epochs", "ppl", "ppl_over_fp", "delta_nll",
             "params", "removed_params_pct", "size_mib_est", "saved_pct_est"]].round(4).to_string(index=False))

w8 = llm.loc[llm["method"].eq("RTN W8 MLP")].iloc[0]
pruning_rows = llm.loc[llm["method"].eq("Magnitude MLP ×5")]
closest = pruning_rows.loc[(pruning_rows["payload_estimate_bytes"] -
                           w8["payload_estimate_bytes"]).abs().idxmin()]
assert closest["ratio"] == .5

                 method  ratio  bits  ft_epochs         ppl  ppl_over_fp  delta_nll    params  removed_params_pct  size_mib_est  saved_pct_est
                     FP   0.00   NaN          0     18.3869       1.0000     0.0000 134515008              0.0000      256.5670         0.0000
             RTN W8 MLP   0.00   8.0          0     18.4008       1.0008     0.0008 134515008              0.0000      182.5411        28.8525
             RTN W4 MLP   0.00   4.0          0     21.0699       1.1459     0.1362 134515008              0.0000      144.5724        43.6512
RTN W4 all block Linear   0.00   4.0          0     23.5349       1.2800     0.2468 134515008              0.0000      107.2628        58.1931
       Magnitude MLP ×5   0.02   NaN          0    278.5340      15.1485     2.7179 132921792              1.1844      253.5282         1.1844
       Magnitude MLP ×5   0.10   NaN          0   5673.0969     308.5394     5.7318 126552384              5.9195      241.3795         5.9195

## Сводный график

Для LeNet круги обозначают варианты без дополнительных эпох, ромбы — с десятью эпохами. Значение ускорения выше 1 означает меньшую задержку, чем у плотной модели. Для SmolLM используется логарифмическая ось, а качество выражено отношением к исходной PPL. Выше 1 означает ухудшение.

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), layout="constrained")
series = cv["method"].where(~cv["method"].str.startswith("PTQ"), "INT8 PTQ")
for name, rows in cv.groupby(series, sort=False):
    axes[0].scatter(rows["size_kib"], rows["accuracy_pct"], label=name, s=48,
                    marker="D" if rows["ft_epochs"].iat[0] else "o")
axes[0].set(title="LeNet · качество и размер", xlabel="Файл state_dict, КиБ", ylabel="Top-1, %")
axes[0].legend(fontsize=8)
selected = pd.DataFrame([ptq, iterative, recovered, dense_ft])
for batch in (1, 256):
    axes[1].plot(selected["method"], selected[f"speedup_b{batch}"], "o-", label=f"Batch {batch}")
axes[1].set_xticks(range(len(selected)), ["INT8", "LAMP", "LAMP + FT", "Dense + FT"])
axes[1].axhline(1, color="gray", linestyle="--")
axes[1].set(title="LeNet · задержка", ylabel="Ускорение относительно FP32, ×")
axes[1].legend()

ax = axes[2]
for name, rows in llm.groupby("method", sort=False):
    ax.scatter(rows["saved_pct_est"], rows["ppl_over_fp"], label=name, s=48)
ax.set(title="SmolLM · качество и размер", xlabel="Экономия расчётной нагрузки, %",
       ylabel="PPL / PPL FP16", yscale="log")
ax.axhline(1, color="gray", linestyle="--")
ax.legend(fontsize=8)
plt.show()

## 3. Дополнительные серии и границы выводов

- **Прунинг LeNet D01–D03.** Неструктурные маски, одношаговое удаление каналов и итеративное восстановление сравниваются со своим бейзлайном. В D03 строка `ratio=0` уже включает дообучение. Ранние исходные веса отличаются от общего CV-прогона. [История и таблицы](cv.ipynb)
- **Квантование D09.** W4G128 на WikiText-2 validation, окно 1024, шаг 512, другие модели и токенизаторы. AWQ показал меньшую PPL на SmolLM2, GPTQ — на Pythia и OPT. Эти строки полезны для выбора метода внутри модели. [Полные результаты и протокол D09](../quantization/research/D09/README.md)
- **Прунинг SmolLM D04–D05.** LAMP, Taylor и Magnitude с разными режимами восстановления дают контекст. Изменение критерия одновременно с добавлением LoRA не изолирует эффект обучения. В основном согласованном LLM-прогоне бюджет дообучения нулевой. [D04](../pruning/description_res/description_d4.md) · [D05](../pruning/description_res/description_d5.md)

### ResNet-20 · CIFAR-10

Отдельная серия структурного L1-прунинга. Ниже используются локальные CSV, когда завершённый запуск доступен. При нескольких завершённых запусках выбирается последний по UTC-имени каталога. Сопоставимого PTQ-прогона в этой серии нет. [Отчёт D06](../pruning/description_res/description_d6.md)

In [5]:
completed = []
for path in sorted((ROOT / "pruning/logs/d06").glob("*/metrics.csv")):
    epochs_path = path.with_name("epochs.csv")
    config_path = path.with_name("config.json")
    if not (epochs_path.is_file() and config_path.is_file()):
        continue
    config = json.loads(config_path.read_text(encoding="utf-8"))
    metrics = pd.read_csv(path)
    epochs = pd.read_csv(epochs_path)
    steps = len(config["targets"])
    n_recovery = steps * config["recovery_epochs_per_step"]
    if (len(metrics) == 1 + 3 * steps and
            epochs["branch"].value_counts().to_dict() ==
            {"base": config["base_epochs"], "pruned": n_recovery, "dense": n_recovery}):
        completed.append((path, metrics))

if completed:
    path, resnet = completed[-1]
    initial = resnet.loc[resnet["phase"].eq("trained")].iloc[0]
    controls = resnet.loc[resnet["branch"].eq("dense")].set_index("step")["test_acc_pct"]
    resnet["delta_dense_pp"] = resnet["test_acc_pct"] - resnet["step"].map(controls)
    for metric in ("params", "macs"):
        resnet[f"saved_{metric}_pct"] = 100 * (1 - resnet[metric] / initial[metric])
    print(path.relative_to(ROOT))
    print(resnet[["step", "phase", "target_channel_removal", "test_acc_pct", "delta_dense_pp",
                    "saved_params_pct", "saved_macs_pct", "latency_b1_ms", "latency_b128_ms"]].round(3).to_string(index=False))
else:
    print("Полные локальные CSV отсутствуют. См. отчёт D06.")

pruning/logs/d06/20261120T105403773154Z/metrics.csv
 step           phase  target_channel_removal  test_acc_pct  delta_dense_pp  saved_params_pct  saved_macs_pct  latency_b1_ms  latency_b128_ms
    0         trained                   0.000         89.67             NaN             0.000           0.000          0.652            1.225
    1 before_recovery                   0.125         79.04          -10.29            12.292          12.284          0.707            1.218
    1  after_recovery                   0.125         88.75           -0.58            12.292          12.284          0.713            1.150
    1  matched_budget                   0.000         89.33            0.00             0.000           0.000          0.672            1.225
    2 before_recovery                   0.250         79.68           -9.75            24.584          24.568          0.694            1.351
    2  after_recovery                   0.250         88.43           -1.00            24.584   

## Выводы по сохранённым измерениям

In [6]:
print(
    f"LeNet, 0 дополнительных эпох. INT8 PTQ: {ptq['accuracy_pct']:.2f}% "
    f"при {ptq['size_kib']:.2f} КиБ. Пятишаговый LAMP: {iterative['accuracy_pct']:.2f}% "
    f"при {iterative['size_kib']:.2f} КиБ. One-shot LAMP на той же доле: "
    f"{oneshot['accuracy_pct']:.2f}% при {oneshot['size_kib']:.2f} КиБ."
)
print(
    f"LeNet, {int(recovered['ft_epochs'])} дополнительных эпох. "
    f"LAMP: {recovered['accuracy_pct']:.2f}%, плотный контроль: {dense_ft['accuracy_pct']:.2f}%. "
    f"Прирост LAMP после дообучения: "
    f"{recovered['accuracy_pct'] - iterative['accuracy_pct']:+.2f} п. п. "
    f"Разница с плотным контролем: {recovered['delta_dense_ft_pp']:+.2f} п. п."
)
print(
    f"LeNet, медиана CPU-задержки FP32 → INT8. "
    f"Batch 1: {cv['b1_ms'].iloc[0]:.2f} → {ptq['b1_ms']:.2f} мс. "
    f"Batch 256: {cv['b256_ms'].iloc[0]:.2f} → {ptq['b256_ms']:.2f} мс."
)
print(
    f"SmolLM, 0 дополнительных эпох. FP16: PPL {llm_base['ppl']:.2f}. "
    f"RTN W8 MLP: PPL {w8['ppl']:.2f} при {w8['size_mib_est']:.2f} МиБ. "
    f"Ближайший по расчётному размеру Magnitude MLP: "
    f"PPL {closest['ppl']:,.2f} при {closest['size_mib_est']:.2f} МиБ."
)
print("Размер SmolLM расчётный. Для RTN измерено качество восстановленных весов.")

LeNet, 0 дополнительных эпох. INT8 PTQ: 98.52% при 56.42 КиБ. Пятишаговый LAMP: 76.17% при 55.98 КиБ. One-shot LAMP на той же доле: 77.75% при 54.98 КиБ.
LeNet, 10 дополнительных эпох. LAMP: 99.04%, плотный контроль: 99.28%. Прирост LAMP после дообучения: +22.87 п. п. Разница с плотным контролем: -0.24 п. п.
LeNet, медиана CPU-задержки FP32 → INT8. Batch 1: 0.28 → 0.59 мс. Batch 256: 8.64 → 4.21 мс.
SmolLM, 0 дополнительных эпох. FP16: PPL 18.39. RTN W8 MLP: PPL 18.40 при 182.54 МиБ. Ближайший по расчётному размеру Magnitude MLP: PPL 745,667.23 при 180.63 МиБ.
Размер SmolLM расчётный. Для RTN измерено качество восстановленных весов.
